In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [2]:
import pandas as pd
import json

file_path = "/home/user/Downloads/bayut_properties_rent_commercial_2026_08_05.json"

df = pd.read_json(file_path)

print(df.shape)
print(df.head())


(29818, 99)
                                            listing_url        id  objectID  \
0  https://www.bayut.com/property/details-16007186.html  12518230  12518230   
1  https://www.bayut.com/property/details-16005899.html  12517047  12517047   
2  https://www.bayut.com/property/details-16005799.html  12516933  12516933   
3  https://www.bayut.com/property/details-15979286.html  12492111  12492111   
4  https://www.bayut.com/property/details-15871970.html  12388917  12388917   

   ownerID  userExternalID  sourceID   state  \
0  2800500         2800500         1  active   
1  2800500         2800500         1  active   
2  2800500         2800500         1  active   
3  2800500         2800500         1  active   
4  2372946         2372946         1  active   

                                          geography   purpose  price  \
0  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent   1800   
1  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent  40000   
2  {'lat

In [14]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse

# ==========================
# LOAD DATA
# ==========================
# df = pd.read_json("your_file.json")
# or
# df = pd.read_json("your_file.json", lines=True)

# ==========================
# BASIC INFO
# ==========================
print("="*80)
print("DATASET SUMMARY")
print("="*80)

print(f"Rows    : {len(df):,}")
print(f"Columns : {len(df.columns)}")

print("\nColumns:")
print(df.columns.tolist())

# ==========================
# EMPTY COLUMN CHECK
# ==========================
print("\n"+"="*80)
print("COMPLETELY EMPTY COLUMNS")
print("="*80)

empty_cols = df.columns[df.isna().all()].tolist()

if empty_cols:
    print(*empty_cols, sep="\n")
else:
    print("No completely empty columns")

# ==========================
# NULL SUMMARY
# ==========================
print("\n"+"="*80)
print("NULL VALUE SUMMARY")
print("="*80)

null_summary = pd.DataFrame({
    "Null Count":df.isna().sum(),
    "Null %":round(df.isna().mean()*100,2)
})

print(null_summary[null_summary["Null Count"]>0].sort_values("Null Count",ascending=False))

# ==========================
# EMPTY STRING SUMMARY
# ==========================
print("\n"+"="*80)
print("EMPTY STRING SUMMARY")
print("="*80)

empty_string=[]

for c in df.columns:
    if df[c].dtype=="object":
        cnt=df[c].fillna("").astype(str).str.strip().eq("").sum()
        if cnt:
            empty_string.append([c,cnt])

if empty_string:
    print(pd.DataFrame(empty_string,columns=["Column","Empty String Count"]))
else:
    print("No empty strings")

# ==========================
# DUPLICATE RECORDS
# ==========================
print("\n"+"="*80)
print("DUPLICATE RECORDS")
print("="*80)

try:
    dup=df.astype(str).duplicated().sum()
    print("Duplicate Rows :",dup)
except:
    print("Could not check duplicate rows (nested objects present)")

# ==========================
# DUPLICATE IDS
# ==========================
id_cols=["id","objectID","externalID","referenceNumber"]

print("\n"+"="*80)
print("DUPLICATE IDS")
print("="*80)

for c in id_cols:
    if c in df.columns:
        d=df[c].duplicated().sum()
        print(f"{c:20} : {d}")

# ==========================
# DUPLICATE URL
# ==========================
if "listing_url" in df.columns:

    print("\n"+"="*80)
    print("DUPLICATE URLS")
    print("="*80)

    d=df["listing_url"].duplicated().sum()
    print("Duplicate listing_url :",d)

# ==========================
# INVALID URL
# ==========================
print("\n"+"="*80)
print("INVALID URLS")
print("="*80)

def valid_url(x):

    if pd.isna(x):
        return False

    try:
        r=urlparse(str(x))
        return r.scheme in ["http","https"] and r.netloc!=""
    except:
        return False

if "listing_url" in df.columns:

    bad=df[~df["listing_url"].apply(valid_url)]

    if len(bad):
        print(bad[["listing_url"]].head(20))
        print("Total Invalid :",len(bad))
    else:
        print("No invalid URLs")

# ==========================
# CONTROL CHARACTERS
# ==========================
print("\n"+"="*80)
print("CONTROL CHARACTERS")
print("="*80)

pattern=r"[\n\r\t]"

issues=[]

for c in df.select_dtypes("object"):

    mask=df[c].fillna("").astype(str).str.contains(pattern,regex=True)

    if mask.any():
        issues.append([c,mask.sum()])

if issues:
    print(pd.DataFrame(issues,columns=["Column","Rows"]))
else:
    print("No control characters")

# ==========================
# LEADING/TRAILING SPACE
# ==========================
print("\n"+"="*80)
print("LEADING / TRAILING SPACE")
print("="*80)

issues=[]

for c in df.select_dtypes("object"):

    mask=df[c].fillna("").astype(str)!=df[c].fillna("").astype(str).str.strip()

    if mask.any():
        issues.append([c,mask.sum()])

if issues:
    print(pd.DataFrame(issues,columns=["Column","Rows"]))
else:
    print("No leading/trailing spaces")

# ==========================
# PHONE VALIDATION
# ==========================
if "phoneNumber" in df.columns:

    print("\n"+"="*80)
    print("PHONE VALIDATION")
    print("="*80)

    phone=df["phoneNumber"].fillna("").astype(str)

    mask=~phone.str.fullmatch(r"[\d+\-\s()]+") & (phone!="")

    print("Invalid Phones :",mask.sum())

# ==========================
# DATE VALIDATION
# ==========================
print("\n"+"="*80)
print("DATE VALIDATION")
print("="*80)

date_cols=["createdAt","approvedAt","updatedAt","touchedAt","reactivatedAt"]

for c in date_cols:

    if c in df.columns:

        invalid=pd.to_datetime(df[c],errors="coerce").isna() & df[c].notna()

        print(f"{c:20} : {invalid.sum()} invalid")

# ==========================
# PRICE VALIDATION
# ==========================
if "price" in df.columns:

    print("\n"+"="*80)
    print("PRICE VALIDATION")
    print("="*80)

    p=pd.to_numeric(df["price"],errors="coerce")

    print("Null Price :",p.isna().sum())
    print("Zero Price :", (p==0).sum())
    print("Negative Price :", (p<0).sum())

# ==========================
# AREA VALIDATION
# ==========================
if "area" in df.columns:

    print("\n"+"="*80)
    print("AREA VALIDATION")
    print("="*80)

    a=pd.to_numeric(df["area"],errors="coerce")

    print("Null :",a.isna().sum())
    print("Zero :", (a==0).sum())
    print("Negative :", (a<0).sum())

# ==========================
# ROOM VALIDATION
# ==========================
for c in ["rooms","baths"]:

    if c in df.columns:

        print("\n"+"="*80)
        print(c.upper())
        print("="*80)

        print(df[c].value_counts(dropna=False).head(20))

# ==========================
# PHOTO COUNT
# ==========================
if "photos" in df.columns and "photoCount" in df.columns:

    print("\n"+"="*80)
    print("PHOTO COUNT VALIDATION")
    print("="*80)

    calc=df["photos"].apply(lambda x:len(x) if isinstance(x,list) else 0)

    print("Mismatch :", (calc!=df["photoCount"]).sum())

# ==========================
# VIDEO COUNT
# ==========================
if "videos" in df.columns and "videoCount" in df.columns:

    calc=df["videos"].apply(lambda x:len(x) if isinstance(x,list) else 0)

    print("\nVideo Count Mismatch :", (calc!=df["videoCount"]).sum())

# ==========================
# PANORAMA COUNT
# ==========================
if "panoramas" in df.columns and "panoramaCount" in df.columns:

    calc=df["panoramas"].apply(lambda x:len(x) if isinstance(x,list) else 0)

    print("Panorama Count Mismatch :", (calc!=df["panoramaCount"]).sum())

# ==========================
# BOOLEAN VALIDATION
# ==========================
print("\n"+"="*80)
print("BOOLEAN VALIDATION")
print("="*80)

bool_cols=[
"active","hidePrice","isVerified","directFromOwner",
"isShortTermRental","isHotelApartment",
"hasExactGeography","isBusinessCenter"
]

for c in bool_cols:

    if c in df.columns:

        bad=~df[c].isin([True,False,np.nan])

        print(f"{c:25} : {bad.sum()} invalid")

print("\n"+"="*80)
print("QA COMPLETED")
print("="*80)

DATASET SUMMARY
Rows    : 29,818
Columns : 99

Columns:
['listing_url', 'id', 'objectID', 'ownerID', 'userExternalID', 'sourceID', 'state', 'geography', 'purpose', 'price', 'product', 'productLabel', 'productVariant', 'rentFrequency', 'referenceNumber', 'permitNumber', 'projectNumber', 'title', 'title_l1', 'title_l2', 'title_l3', 'title_l4', 'title_l5', 'title_l6', 'title_l7', 'title_l8', 'description', 'description_l1', 'description_l2', 'description_l3', 'description_l4', 'description_l5', 'description_l6', 'description_l7', 'description_l8', 'descriptionTranslated', 'descriptionTranslated_l1', 'descriptionTranslated_l2', 'descriptionTranslated_l3', 'externalID', 'slug', 'location', 'category', 'createdAt', 'approvedAt', 'updatedAt', 'touchedAt', 'reactivatedAt', 'rooms', 'baths', 'area', 'score', 'score_l1', 'score_l2', 'score_l3', 'coverPhoto', 'photoCount', 'videoCount', 'panoramaCount', 'photos', 'floorPlans', 'videos', 'panoramas', 'amenities', 'phoneNumber', 'contactMethodAvail

In [ ]:
# import pandas as pd
# import json

# df = pd.read_json(
#     r"/home/user/Downloads/haraj_kse_2026_07.json",
#     lines=True
# )

# # Convert list/dict columns into strings
# df = df.apply(
#     # lambda col: col.map(
#         lambda x: json.dumps(x, sort_keys=True)
#         if isinstance(x, (list, dict))
#         else x
#     )
# )

In [15]:
import json

file_path = "/home/user/Downloads/bayut_properties_rent_commercial_2026_08_05.json"

invalid_records = []

with open(file_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            invalid_records.append({
                "Record": line_no,
                "Line": e.lineno,
                "Column": e.colno,
                "Character": e.pos,
                "Error": e.msg
            })

if invalid_records:
    print(f"\n❌ Found {len(invalid_records)} invalid JSON record(s)\n")
    for r in invalid_records:
        print(
            f"Record {r['Record']} | "
            f"Line {r['Line']} | "
            f"Column {r['Column']} | "
            f"Character {r['Character']} | "
            f"{r['Error']}"
        )
else:
    print("✅ All JSON records are valid.")


❌ Found 29818 invalid JSON record(s)

Record 1 | Line 1 | Column 28178 | Character 28177 | Expecting value
Record 2 | Line 1 | Column 31125 | Character 31124 | Extra data
Record 3 | Line 1 | Column 29672 | Character 29671 | Extra data
Record 4 | Line 1 | Column 29547 | Character 29546 | Extra data
Record 5 | Line 1 | Column 28258 | Character 28257 | Extra data
Record 6 | Line 1 | Column 32888 | Character 32887 | Extra data
Record 7 | Line 1 | Column 34189 | Character 34188 | Extra data
Record 8 | Line 1 | Column 25375 | Character 25374 | Extra data
Record 9 | Line 1 | Column 20569 | Character 20568 | Extra data
Record 10 | Line 1 | Column 17415 | Character 17414 | Extra data
Record 11 | Line 1 | Column 33249 | Character 33248 | Extra data
Record 12 | Line 1 | Column 33665 | Character 33664 | Extra data
Record 13 | Line 1 | Column 20734 | Character 20733 | Extra data
Record 14 | Line 1 | Column 22962 | Character 22961 | Extra data
Record 15 | Line 1 | Column 21302 | Character 21301 | E

In [ ]:
import json

file_path = "/home/user/Downloads/bayut_properties_rent_commercial_2026_08_05.json"

invalid_records = []

with open(file_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            continue

        try:
            json.loads(line)
        except json.JSONDecodeError as e:
            invalid_records.append({
                "Line": line_no,
                "Column": e.colno,
                "Character": e.pos,
                "Issue": e.msg,
                "Content": line[:250] + ("..." if len(line) > 250 else "")
            })

if invalid_records:
    print(f"\n❌ Found {len(invalid_records)} invalid JSON record(s)\n")

    for rec in invalid_records:
        print("="*100)
        print(f"Line      : {rec['Line']}")
        print(f"Column    : {rec['Column']}")
        print(f"Character : {rec['Character']}")
        print(f"Issue     : {rec['Issue']}")
        print("\nRecord:")
        print(rec["Content"])
        print("="*100)

else:
    print("✅ File contains only valid JSON records.")

In [ ]:
# req_cols = """id, url, broker_display_name, broker, category, category_url, title, property_type, sub_category_1, sub_category_2, description, location, depth, price, currency, price_per, bedrooms, bathrooms, furnished, rera_permit_number, dtcm_licence, scraped_ts, amenities, details, agent_name, reference_number, number_of_photos, user_id, published_at, phone_number, date, iteration_number, latitude, longitude"""

# req_cols = [c.strip() for c in req_cols.split(",") if c.strip()]

# missing = [c for c in req_cols if c not in df.columns]
# extra = [c for c in df.columns if c not in req_cols]

# print("Requirement columns:", len(req_cols))
# print("File columns:", len(df.columns))
# print("Missing:", missing)
# print("Extra:", extra)
# print("Order matches:", req_cols == list(df.columns))

Requirement columns: 34
File columns: 34
Missing: []
Extra: []
Order matches: True


In [6]:
import pandas as pd
import json

df1 = df.copy()

for col in df1.columns:
    df1[col] = df1[col].apply(
        lambda x: json.dumps(x, sort_keys=True) if isinstance(x, (dict, list)) else x
    )

print("Duplicate rows:", df1.duplicated().sum())

# View duplicate rows
duplicates = df1[df1.duplicated(keep=False)]
print(duplicates)

Duplicate rows: 0
Empty DataFrame
Columns: [listing_url, id, objectID, ownerID, userExternalID, sourceID, state, geography, purpose, price, product, productLabel, productVariant, rentFrequency, referenceNumber, permitNumber, projectNumber, title, title_l1, title_l2, title_l3, title_l4, title_l5, title_l6, title_l7, title_l8, description, description_l1, description_l2, description_l3, description_l4, description_l5, description_l6, description_l7, description_l8, descriptionTranslated, descriptionTranslated_l1, descriptionTranslated_l2, descriptionTranslated_l3, externalID, slug, location, category, createdAt, approvedAt, updatedAt, touchedAt, reactivatedAt, rooms, baths, area, score, score_l1, score_l2, score_l3, coverPhoto, photoCount, videoCount, panoramaCount, photos, floorPlans, videos, panoramas, amenities, phoneNumber, contactMethodAvailability, contactName, agency, active, hasExactGeography, verification, isVerified, furnishingStatus, extraFields, type, completionStatus, agentA

In [7]:
df.id.duplicated().sum()

np.int64(0)

In [4]:
import re

url_pattern = re.compile(
    r'^(https?://)'
    r'([a-zA-Z0-9-]+\.)+[a-zA-Z]{2,}'
    r'(/[^\s]*)?$'
)

invalid_url = df[
    ~df["Website"].fillna("").astype(str).str.match(url_pattern)
]

print(f"Invalid URLs: {len(invalid_url)}")
display(invalid_url[[ "Website"]])

Invalid URLs: 10002


,Website
0,www.sevencentury.com
1,onplan.ae
2,www.harbordubai.com
3,NaN
4,NaN
5,Mohammad Rafie
6,NaN
7,www.a1properties.ae
8,Mahoud Khalil
9,NaN


In [10]:
null_counts = df.isnull().sum().sort_values(ascending=False)
print(null_counts)

projectNumber                29818
floorPlan                    29817
clips                        29816
isHotelApartment             29809
plotArea                     29794
isShortTermRental            29772
descriptionTranslated_l3     29718
availabilityStatus           29713
directFromOwner              29686
descriptionTranslated_l2     29665
coverVideo                   29468
completionDetails            29137
paymentPlans                 29129
paymentPlanSummaries         29129
project                      28423
productVariant               26599
isBusinessCenter             23115
occupancyStatus              17273
furnishingStatus              4706
permitNumber                  3114
baths                          338
descriptionTranslated_l1       180
coverPhoto                       8
description_l8                   2
description_l7                   2
description_l6                   2
description_l4                   2
description_l5                   2
price               

In [9]:
import re

# URL validation regex
url_pattern = re.compile(
    r"^(https?://)?([A-Za-z0-9-]+\.)+[A-Za-z]{2,}(/.*)?$",
    re.IGNORECASE
)

# Remove nulls and empty strings
website = df["Website"].dropna().astype(str).str.strip()
website = website[website != ""]

# Find invalid websites
invalid = df.loc[website.index][~website.str.match(url_pattern)]

# Add Excel row numbers
invalid = invalid.copy()
invalid["row_number"] = invalid.index + 2

print("Invalid website values:", len(invalid))
print(invalid[["row_number", "Website"]])

KeyError: 'Website'

In [12]:
df.head(10)

listing_url        id  objectID  \
0  https://www.bayut.com/property/details-16007186.html  12518230  12518230   
1  https://www.bayut.com/property/details-16005899.html  12517047  12517047   
2  https://www.bayut.com/property/details-16005799.html  12516933  12516933   
3  https://www.bayut.com/property/details-15979286.html  12492111  12492111   
4  https://www.bayut.com/property/details-15871970.html  12388917  12388917   
5  https://www.bayut.com/property/details-15872535.html  12389447  12389447   
6  https://www.bayut.com/property/details-15872336.html  12389260  12389260   
7  https://www.bayut.com/property/details-16005576.html  12516727  12516727   
8  https://www.bayut.com/property/details-13617534.html  10226983  10226983   
9  https://www.bayut.com/property/details-13010089.html   9680024   9680024   

   ownerID  userExternalID  sourceID   state  \
0  2800500         2800500         1  active   
1  2800500         2800500         1  active   
2  2800500         2800500         1  active   
3  2800500         2800500         1  active   
4  2372946         2372946         1  active   
5  2372946         2372946         1  active   
6  2372946         2372946         1  active   
7  2800500         2800500         1  active   
8  2372946         2372946         1  active   
9  2372946         2372946         1  active   

                                          geography   purpose  price  \
0  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent   1800   
1  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent  40000   
2  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent  35000   
3  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent   6500   
4  {'lat': 25.245609667644, 'lng': 55.311674431843}  for-rent  15000   
5  {'lat': 25.245609667644, 'lng': 55.311674431843}  for-rent  25000   
6  {'lat': 25.245609667644, 'lng': 55.311674431843}  for-rent  12000   
7  {'lat': 25.258404750499, 'lng': 55.298483669758}  for-rent  11000   
8  {'lat': 25.245609667644, 'lng': 55.311674431843}  for-rent  27000   
9  {'lat': 25.245609667644, 'lng': 55.311674431843}  for-rent   3000   

    product productLabel     productVariant rentFrequency referenceNumber  \
0  superhot      default  Signature 14 days        yearly          EJ01FH   
1  superhot      default  Signature 14 days        yearly         OFF22FH   
2  superhot      default  Signature 14 days        yearly         OFF13FH   
3  superhot      default  Signature 14 days        yearly       FLEXI01FH   
4  superhot      default  Signature 30 days        yearly       ALGURG-05   
5  superhot      default  Signature 30 days        yearly       ALGURG-04   
6  superhot      default  Signature 30 days        yearly       ALGURG-06   
7  superhot      default  Signature 14 days        yearly        DESK01FH   
8  superhot      default  Signature 30 days        yearly        ABN-7851   
9  superhot      default  Signature 30 days        yearly          C-001B   

  permitNumber  projectNumber  \
0         None            NaN   
1         None            NaN   
2         None            NaN   
3         None            NaN   
4         None            NaN   
5         None            NaN   
6         None            NaN   
7         None            NaN   
8         None            NaN   
9         None            NaN   

                                                                                          title  \
0                                    Virtual Office | Ejari | Prime Business Address | AED 1800   
1                   Prime Office Space for Rent | 230 Sq. Ft. | Arabian Square | Close to Metro   
2                                  Affordable Furnished Office | 200 Sq. Ft. | Flexible Payment   
3                                            Flexi Desk | Ejari | Business Address | Near Metro   
4                          High‑Value Executive Office | Fully Furnished + DEWA & WiFi Included   
5                                          Pre

In [13]:
phone_pattern = r"^[+]?[0-9()\-\s]+$"

invalid_phone = df[
    df["Phone Number"].notna() &
    ~df["Phone Number"].astype(str).str.fullmatch(phone_pattern)
]

print("Invalid Phone Numbers:", len(invalid_phone))
print(invalid_phone[["Phone Number"]])

Invalid Phone Numbers: 108
         Phone Number
3       971|501151118
64      971|504264267
116     971|504694448
152      971|44225079
184     971|528550000
186     971|543131168
207      971|43682168
250     971|558559751
251      971|46969499
278     971|042545070
291     971|509154354
340     971|585306318
365     971|529109324
441     971|559361112
521     971|526699625
542     971|506285858
572     971|555501477
625     971|562212721
631     971|529948887
682     971|509533066
693      971|42396655
917     971|503442133
945     971|527513222
1020    971|567512389
1065    971|528588118
1184    971|522437443
1192    971|586492729
1235    971|556698774
1298     971|44546695
1353    971|504815644
1383    971|506909809
1483    971|529568060
1652     971|42699113
1655     971|44385044
1679    971|508244492
1803    971|561883983
1866    971|043210201
2036    971|521940000
2080    971|501652929
2175    971|567200137
2189     971|42253778
2207    971|549957499
2337    971|551045220
2377 

In [14]:
import re

phone_pattern = r"^971\|(0?[1-9]\d{7,9})$"

invalid_phone = df[
    df["Phone Number"].notna() &
    ~df["Phone Number"].astype(str).str.fullmatch(phone_pattern)
]

print(invalid_phone[["Phone Number"]])

            Phone Number
0          971-4-4520077
2        971-971-3251616
4          971-4-4297040
5          971-4-3298844
6            97142438000
7          971-4-3515583
8         971-04-3990990
9          971-4-4522202
10             044473277
11         971-4-3026000
12           97144227900
13         971-4-2212442
14         971-4-3236222
15             043499767
16         971-4-4391200
17         971-4-2957771
18         971-4-2500177
19         971-4-3532000
20             043663334
21         971-4-3430111
22           97145616900
23             043518800
24         971-4-3552266
25         971-4-5516626
26            0585299999
27           97143799919
29         971-4-2661167
30        971-04-3691774
32         971-4-3631999
33         971-4-4472730
34           97144323920
35           97144207170
36         971-4-4524466
37         971-4-5539020
38         971-4-2977302
39         971-4-4473501
40         971-4-3295959
41             044328398
42         971-4-3476853


In [6]:
df.isnull().sum()

Office Number       0
Name English        0
Name Arabic         0
Website          9424
Phone Number     7880
Email              25
dtype: int64

In [15]:
import pandas as pd

df1 = df.replace(r'^\s*$', pd.NA, regex=True)

empty_columns = df1.columns[df1.isna().all()]

print("Completely empty columns:")
print(list(empty_columns))

Completely empty columns:
['sub_category_2', 'depth', 'price', 'currency', 'price_per', 'bedrooms', 'bathrooms', 'furnished', 'rera_permit_number', 'dtcm_licence', 'amenities', 'details', 'agent_name', 'reference_number', 'phone_number', 'latitude', 'longitude']


In [19]:
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

invalid_email = df[
    df["Email"].notna() &
    ~df["Email"].astype(str).str.fullmatch(email_pattern)
]

print("Invalid Emails:", len(invalid_email))
print(invalid_email[["Email"]])

Invalid Emails: 0
Empty DataFrame
Columns: [Email]
Index: []


In [3]:
df.iteration_number.value_counts()

iteration_number
202608    1141
Name: count, dtype: int64

In [12]:
df.id.duplicated().sum()

np.int64(0)

In [11]:
df[df.url.duplicated() >= 1]

,id,url,broker_display_name,broker,category,category_url,title,property_type,sub_category_1,sub_category_2,description,location,depth,price,currency,price_per,bedrooms,bathrooms,furnished,rera_permit_number,dtcm_licence,scraped_ts,amenities,details,agent_name,reference_number,number_of_photos,user_id,published_at,phone_number,date,iteration_number,latitude,longitude
19972,11183753030,https://haraj.com.sa/11183754057/شقة_مفروشه_فاخرة_للايجار_في_أبها,x0x507,X0X507,rent,/tags/%D8%B4%D9%82%D9%82%20%D9%84%D9%84%D8%A7%D9%8A%D8%AC%D8%A7%D8%B1,احتاج شقة في حي النخيل نظيفه وايجارها معقول وكبيره,Apartments,Residential,,احتاج شقة في حي النخيل نظيفه وايجارها معقول وكبيره,أبها,,,,,,,,,,2026-07-27,,,,,1,x0x507,2026-07-04,,2026-07-27,202608,,


In [10]:
df.url.duplicated().sum()

np.int64(1)

In [7]:
df.broker_licenses.value_counts()

broker_licenses
[{"end_date": "08/11/2026", "license_number": "36806", "license_type": "Trade License"}]            1
[{"end_date": "31/03/2027", "license_number": "5606", "license_type": "Trade License"}]             1
[{"end_date": "19/02/2027", "license_number": "64891", "license_type": "Trade License"}]            1
[{"end_date": "02/09/2026", "license_number": "7921", "license_type": "Trade License"}]             1
[{"end_date": "03/02/2027", "license_number": "70776", "license_type": "Trade License"}]            1
[{"end_date": "01/09/2026", "license_number": "38571", "license_type": "Trade License"}]            1
[{"end_date": "28/01/2027", "license_number": "116496", "license_type": "Trade License"}]           1
[{"end_date": "15/02/2027", "license_number": "2800", "license_type": "Trade License"}]             1
[{"end_date": "12/04/2027", "license_number": "130893", "license_type": "Professional License"}]    1
[{"end_date": "07/07/2027", "license_number": "119036", "license_t

In [9]:
df[df.broker_licenses == "[{}]""]

SyntaxError: unterminated string literal (detected at line 1) (190675803.py, line 1)

In [10]:
df[df["broker_licenses"].astype(str) == "[{}]"]

,unique_id,url,name,license_number,license_expiry_date,broker_type,mobile_number,email_address,the_unified_number_of_the_establishment,broker_licenses,street,city,region,district,building_number,additional_number,scraped_ts,date,iteration_number
155,c157d8a0-3bfb-4779-a68c-39c9c60f16b6,https://eservices.ajmanded.ae/en/TradeLicense/License/c157d8a0-3bfb-4779-a68c-39c9c60f16b6,YOUR CHOICE REALESTATE INVES L.L.C,114813,13/10/2026,,,,,[{}],,,Aamra,,ألعامرة 1,5,2026-07-29,2026-07-29,202608


In [11]:
df.name.value_counts()

name
.BIN MALEK REAL ESTATE/ (S.P.S - L.L.C)                                                                                        1
ALBAIT ALFARID REALSTATE                                                                                                       1
AL MOTAKAMLA REALSTATE                                                                                                         1
AL FAISAL REALESTATE SERVICES                                                                                                  1
ALMUNA  REALSTATE-L.L.C                                                                                                        1
ROYAL PALAC REAL ESTATE (S.P.S - L.L.C)                                                                                        1
AL HAYAT REALESTATE L.L.C                                                                                                      1
YOUSEF BIN NASSER & SONS REAL ESTATE (S.P.S - L.L.C)                                        

In [18]:
import pandas as pd

text_cols = df.select_dtypes(include="object").columns

# Regex for whitespace issues
pattern = r"(^ )|( $)|( {2,})|(\t)|(\n)|(\r)|(\u00A0)"

for col in text_cols:
    mask = (
        df[col]
        .fillna("")
        .astype(str)
        .str.contains(pattern, regex=True)
    )

    if mask.any():
        print(f"\n=== {col}: {mask.sum()} rows ===")

        cols = ["website" col]
        cols = [c for c in cols if c in df.columns]

        print(
            df.loc[mask, cols]
              .reset_index(names="row_no")
        )

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2854391279.py, line 19)

In [13]:
df.isna().sum()

id                     0
url                    0
broker_display_name    0
broker                 0
category               0
category_url           0
title                  0
property_type          0
sub_category_1         0
sub_category_2         0
description            0
location               0
depth                  0
price                  0
currency               0
price_per              0
bedrooms               0
bathrooms              0
furnished              0
rera_permit_number     0
dtcm_licence           0
scraped_ts             0
amenities              0
details                0
agent_name             0
reference_number       0
number_of_photos       0
user_id                0
published_at           0
phone_number           0
date                   0
iteration_number       0
latitude               0
longitude              0
dtype: int64

In [14]:
(df == "").sum()

id                         0
url                        0
broker_display_name        0
broker                     0
category                   0
category_url               0
title                      0
property_type              0
sub_category_1             1
sub_category_2         32356
description               20
location                   0
depth                  32356
price                  32356
currency               32356
price_per              32356
bedrooms               32356
bathrooms              32356
furnished              32356
rera_permit_number     32356
dtcm_licence           32356
scraped_ts                 0
amenities              32356
details                32356
agent_name             32356
reference_number       32356
number_of_photos        6223
user_id                    0
published_at               0
phone_number           32356
date                       0
iteration_number           0
latitude               32356
longitude              32356
dtype: int64

In [18]:
df[df.license_expiry_date == ""]

,unique_id,url,name,license_number,license_expiry_date,broker_type,mobile_number,email_address,the_unified_number_of_the_establishment,broker_licenses,street,city,region,district,building_number,additional_number,scraped_ts,date,iteration_number
111,ac2635f3-cf27-4793-82c7-595f0334c793,https://eservices.ajmanded.ae/en/TradeLicense/License/ac2635f3-cf27-4793-82c7-595f0334c793,AL QUDRAT REALESTATE - L.L.C,67751,,,,,,"[{""end_date"": ""10/03/2027"", ""license_number"": ""67751"", ""license_type"": ""Professional License""}]",,,Rashideya 1,,Horizon tower,610,2026-07-29,2026-07-29,202608


In [19]:
import pandas as pd
import re

# Column to check
col = "name"   # change to your column name

# Expected keywords
keywords = [
    "Real estate",
    "Realestate",
    "Realstate",
    "Real estates",
    "Realty",
    "Properties",
    "Property",
    "Property management",
    "Realtors",
    "Business Center",
    "Holiday Homes",
    "Hotel Apartments"
]

# Normalize text
series = df[col].fillna("").astype(str)

print("===== Expected Keywords =====")

matched_rows = pd.Series(False, index=df.index)

for kw in keywords:
    mask = series.str.contains(re.escape(kw), case=False, regex=True)
    matched_rows |= mask
    print(f"{kw:<25}: {mask.sum()}")

print("\n===== Rows with Other Values =====")

others = (
    df.loc[~matched_rows, [col]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Unique other values: {len(others)}")
print(others)

===== Expected Keywords =====
Real estate              : 783
Realestate               : 110
Realstate                : 8
Real estates             : 0
Realty                   : 3
Properties               : 191
Property                 : 18
Property management      : 7
Realtors                 : 0
Business Center          : 16
Holiday Homes            : 0
Hotel Apartments         : 15

===== Rows with Other Values =====
Unique other values: 1
                               name
0  العرب للعقارات ذ.م.م - Branch 01


In [20]:
import pandas as pd
import re

col = "name"

expected = {
    "REAL ESTATE",
    "REALESTATE",
    "REALSTATE",
    "REAL ESTATES",
    "REALTY",
    "PROPERTIES",
    "PROPERTY",
    "PROPERTY MANAGEMENT",
    "REALTORS",
    "BUSINESS CENTER",
    "HOLIDAY HOMES",
    "HOTEL APARTMENTS"
}

# Extract suffix after company name
suffix = (
    df[col]
    .fillna("")
    .str.upper()
    .str.extract(r'([A-Z ]+)$')[0]
    .str.strip()
)

print("===== Expected Types =====")
print(suffix.value_counts()[suffix.value_counts().index.isin(expected)])

print("\n===== Unexpected Types =====")
unexpected = suffix[~suffix.isin(expected)].value_counts()
print(unexpected)

===== Expected Types =====
0
REAL ESTATE    2
Name: count, dtype: int64

===== Unexpected Types =====
0
C                                                             675
LC                                                              3
BR                                                              3
WHITE SPACE REAL ESTATE                                         1
AQAR LINE  REAL ESTATE                                          1
FOUR SEASONS REAL ESTATE                                        1
HATTA REAL ESTATE                                               1
ALMAIMANI  REAL ESTATE                                          1
TILAL ALEASIMA REAL ESTATE DEVELOPMENTS                         1
ALEBTIKAR REAL ESTATE                                           1
QIMMA ALIBDAA  REAL ESTATE                                      1
APEX REAL ESTATE                                                1
AL FURSA AL ZAHABIA REAL ESTATE                                 1
AHMED ALSAADI REAL ESTATE             

In [21]:
df[df.name == " العرب للعقارات ذ.م.م - Branch 01"]

,unique_id,url,name,license_number,license_expiry_date,broker_type,mobile_number,email_address,the_unified_number_of_the_establishment,broker_licenses,street,city,region,district,building_number,additional_number,scraped_ts,date,iteration_number


In [22]:
df[df["name"] == "العرب للعقارات ذ.م.م - Branch 01"]

,unique_id,url,name,license_number,license_expiry_date,broker_type,mobile_number,email_address,the_unified_number_of_the_establishment,broker_licenses,street,city,region,district,building_number,additional_number,scraped_ts,date,iteration_number
877,98bf3b22-e4dc-4048-8827-78312394cf8b,https://eservices.ajmanded.ae/en/TradeLicense/License/98bf3b22-e4dc-4048-8827-78312394cf8b,العرب للعقارات ذ.م.م - Branch 01,139525,12/07/2027,,,,,"[{""end_date"": ""12/07/2027"", ""license_number"": ""139525"", ""license_type"": ""Trade License""}]",,,Jurf 3,,ABDULRAHMAN BUILDING,,2026-07-29,2026-07-29,202608


In [23]:
df.mobile_number.value_counts()

mobile_number
              1140
9098409509       1
Name: count, dtype: int64

In [25]:
mask = (
    df["building_number"].isna() |
    (df["building_number"].astype(str).str.strip() == "")
)

df.loc[mask, ["building_number", "url"]].reset_index(names="row_no")

,row_no,building_number,url
0,8,,https://eservices.ajmanded.ae/en/TradeLicense/License/c17119f1-d37a-4dc8-ba4a-5c9979762ce8
1,28,,https://eservices.ajmanded.ae/en/TradeLicense/License/5ad01373-fecf-4b88-af07-38deda88550d
2,102,,https://eservices.ajmanded.ae/en/TradeLicense/License/c768d1cf-3326-46c8-9ec4-989cfa5c22fb
3,122,,https://eservices.ajmanded.ae/en/TradeLicense/License/0d7b0b87-1489-46f8-8c7b-1441efa6d6a4
4,159,,https://eservices.ajmanded.ae/en/TradeLicense/License/626456e9-a796-462f-bb4c-948c6841231d
5,160,,https://eservices.ajmanded.ae/en/TradeLicense/License/d7dc0425-f38e-4e1d-9e22-50caa6ed8d02
6,191,,https://eservices.ajmanded.ae/en/TradeLicense/License/1e7c91c9-a767-4146-9dde-79f65af8ef16
7,198,,https://eservices.ajmanded.ae/en/TradeLicense/License/4188300e-6f70-479f-9d86-ae04404f7e5b
8,247,,https://eservices.ajmanded.ae/en/TradeLicense/License/48c59116-d636-4506-bf8d-a04eb08355a8
9,271,,https://eservices.ajmanded.ae/en/TradeLicense/License/e8feda51-9b8e-4c74-b2cb-772b82dea9e9


In [30]:
import pandas as pd

# Parse expiry date
df["license_expiry_date"] = pd.to_datetime(
    df["license_expiry_date"],
    dayfirst=True,
    errors="coerce"
)

# Convert iteration_number (202608 -> 2026-08-01)
df["iteration_date"] = pd.to_datetime(
    df["iteration_number"].astype(str).str[:4] + "-" +
    df["iteration_number"].astype(str).str[4:6] + "-01"
)

# Two months before iteration date
df["cutoff_date"] = df["iteration_date"] - pd.DateOffset(months=2)

# Records that should have been excluded
mask = df["license_expiry_date"] < df["cutoff_date"]

result = df.loc[
    mask,
    [
        "unique_id",
        "url",
        "license_expiry_date",
        "iteration_number"
    ]
].reset_index(names="row_no")

print(f"Records that should have been excluded: {len(result)}")
print(result)

Records that should have been excluded: 1
   row_no                             unique_id  \
0     448  09102247-1054-4fc6-87d4-554e06a77ae0   

                                                                                          url  \
0  https://eservices.ajmanded.ae/en/TradeLicense/License/09102247-1054-4fc6-87d4-554e06a77ae0   

  license_expiry_date  iteration_number  
0          2026-05-31            202608  


In [29]:
print(df["iteration_number"].dtype)
print(df["iteration_number"].head())

int64
0    202608
1    202608
2    202608
3    202608
4    202608
Name: iteration_number, dtype: int64


In [17]:
df.duplicated().sum()

np.int64(0)